In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import zipfile, os
zip_path = '/content/drive/MyDrive/kvasir-dataset-v2.zip'
extract_path = '/content/KVASIR DATASET'
print('Unzipping...')
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)
print('Done! ✅')

In [ ]:
import os
inner_path = '/content/KVASIR DATASET/kvasir-dataset-v2'
folders = os.listdir(inner_path)
print('Folders found:', folders)
print('Total classes:', len(folders))

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
print('TensorFlow version:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
DATASET_DIR = '/content/KVASIR DATASET/kvasir-dataset-v2'
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 15
NUM_CLASSES = 8

In [ ]:
train_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2, rotation_range=20, zoom_range=0.2, horizontal_flip=True, shear_range=0.1)
val_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)
train_gen = train_datagen.flow_from_directory(DATASET_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', subset='training', shuffle=True)
val_gen = val_datagen.flow_from_directory(DATASET_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', subset='validation', shuffle=False)
print('Classes:', train_gen.class_indices)
print('Train samples:', train_gen.samples)
print('Val samples:', val_gen.samples)

In [ ]:
base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.4)(x)
output = Dense(NUM_CLASSES, activation='softmax')(x)
model = Model(inputs=base_model.input, outputs=output)
model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
print('Model ready! ✅')

In [ ]:
callbacks = [EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True, verbose=1), ModelCheckpoint('best_model.h5', monitor='val_accuracy', save_best_only=True, verbose=1)]
print('Training started 🚀')
history = model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS, callbacks=callbacks)

In [ ]:
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Val')
plt.title('Accuracy')
plt.legend()
plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Val')
plt.title('Loss')
plt.legend()
plt.savefig('/content/drive/MyDrive/kvasir_graphs.png')
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
val_gen.reset()
y_pred = model.predict(val_gen)
y_pred = np.argmax(y_pred, axis=1)
y_true = val_gen.classes
class_names = list(val_gen.class_indices.keys())
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
model.save('/content/drive/MyDrive/kvasir_final_model.h5')
print('Model saved! ✅')